In [1]:
from pathlib import Path

dir_evals = Path('docs/evals')

In [7]:
import re

def getSectionPairsForComparison(file_name):

    def getSectionAndContent(file_path):

        with open(file_path) as fp:
            data = '\n'.join([line.strip() for line in fp.readlines()])
        
        output = re.findall(r'(#+ .+\n?)([^#]+)', data)

        return output

    original = getSectionAndContent(dir_evals / 'original' / file_name)
    generated = getSectionAndContent(dir_evals / 'outputs' / file_name)

    combined = []

    for (k1, v1), (k2, v2) in zip(original, generated):
        k1, k2 = k1.strip(), k2.strip()
        if k1 != k2:
            print(k1, k2)
            continue

        if k1.startswith('## References'): continue

        combined.append((k1, [v1, v2]))

    header = []
    section_pairs = []
    for s, content in combined:
        if not header:
            header.append(s)
            section_pairs.append((s, content))
            continue
        num_hashes_current_header = len(header[-1].split(' ')[0])
        num_hashes_new_header = len(s.split(' ')[0])
        while header and num_hashes_current_header >= num_hashes_new_header:
            header.pop()
            num_hashes_current_header = len(header[-1].split(' ')[0])

        if num_hashes_current_header < num_hashes_new_header:
            header.append(s)
            section_pairs.append(('\n'.join(header), content))

    return section_pairs


In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain.output_parsers.fix import OutputFixingParser
from pydantic import BaseModel, Field
from src.backend.ai.prompts import setPrompt
from src.backend.ai.common import State
from src.backend.utils import Config

class SummarizeSchema(BaseModel):
    '''
    Returns the summary of the provided content
    '''
    summary: str = Field(description='Summary of the provided content')

# ---------------------------------------------------------------------------
class Summarize:

    summarize_system_prompt = f'''\
    You are an expert on writing summary of a given content.
    <Instructions>
    - The summary must not contain more than {Config.NUM_TOKENS_SUMMARY} tokens. 
    - The summary must include all important aspects of a given content.
    </Instructions>
    '''

    summarize_human_prompt = '''
    <Content>
    {content}
    </Content>
    
    <Instructions>
    - Generate a comprehensive summary of the content.
    
    - Provide the output in the following format.
    {format_instructions}

    - Output must be in JSON format with `json` tags.
    </Instructions>
    '''

    def __init__(self, llm):

        parser = OutputFixingParser.from_llm(parser=PydanticOutputParser(pydantic_object=SummarizeSchema), 
                                             llm=llm,
                                             max_retries=Config.RETRY_COUNTER)
        self.summarize_prompt = setPrompt(self.summarize_system_prompt, self.summarize_human_prompt, parser)
        self.summarize_chain = self.summarize_prompt | llm | parser

    def __call__(self, state: State):
        '''LLM generates summary for a given content'''

        response = self.summarize_chain.invoke(input={'content': state['content_pre']})

        try:
            response = dict(response)['summary']
        except:
            raise Exception(f'Summarize response does not have content, response: {response}')
        
        return {'content_pre': response, 'steps': ['Summarize']}

class ScoreWithReasonSchema(BaseModel):
    '''
    Returns a score within 0 to 100 for a particular criterion and a statement supporting the score
    '''
    score: int = Field(description='Score within 0 to 100')
    reason: str = Field(description='A short statement supporting the score')

class RateContentSchema(BaseModel):
    '''
    Returns scores based on different criteria to rate the content of a structured document
    '''

    relevance: ScoreWithReasonSchema = Field(description='A score with supporting statement that evaluates the Relevance of the content')
    continuity: ScoreWithReasonSchema = Field(description='A score with supporting statement that evaluates the Continuity of the content')
    non_repetitiveness: ScoreWithReasonSchema = Field(description='A score with supporting statement that evaluates the Uniqueness (or non-repetitiveness) of the content')
    specificity: ScoreWithReasonSchema = Field(description='A score with supporting statement that evaluates the Specificity of the content')
    #relevance of citation (hallucination), accuracy of the content+citation (hallucination), specificity of the content

# ---------------------------------------------------------------------------
class RateContent:

    rate_content_system_prompt = '''\
    You will be provided a manuscript section header with or without previously written content summary on a specific topic. You are a scholarly reviewer with expertise in the corresponding topic area. Your task is review and rate the section content.
    
    <Instructions>
    - Rating must be an integer number within 0 (lowest) and 100 (highest).
    - Rating will base on the following criteria.
    **Relevance:** Is the content relevant to the provided section header and the summary (if provided).
    **Continuity:** Is the content writing style is continuous with the previously written content summary. Put the highest score if previously written content summary is not provided.
    **Non-Repetitiveness:** Is the content devoid of repetitiveness compared with the previously written content summary. Put the highest score if previously written content summary is not provided.
    **Specificity:** Is the content devoid of halucination.
    </Instructions>
    '''

    rate_content_human_prompt = '''
    <Previous Content Summary>
    {content_pre}
    </Previous Content Summary>

    <Current Section Header>
    {current_section}
    </Current Section Header>

    <Content>
    {content}
    </Content>
    
    <Instructions>
    - Read the Previous Content Summary. 
    - Review the provided content based on the Current Section Header
    - Provide the output in the following format.
    {format_instructions}
    
    - Output must be in JSON format with `json` tags.
    </Instructions>
    '''

    def __init__(self, llm):

        parser = OutputFixingParser.from_llm(parser=PydanticOutputParser(pydantic_object=RateContentSchema), 
                                             llm=llm,
                                             max_retries=Config.RETRY_COUNTER)
        
        self.rate_content_prompt = setPrompt(self.rate_content_system_prompt, self.rate_content_human_prompt, parser)
        
        self.rate_content_chain = self.rate_content_prompt | llm | parser


    def __call__(self, state: State):
        '''LLM generates reports from a given outline'''
        
        response = self.rate_content_chain.invoke(input={'content_pre': state['content_pre'],
                                                             'current_section': state['current_section'],
                                                             'content': state['content']})
        try:
            content = dict(response)
        except:
            raise Exception(f'GenerateContent response does not have content, response: {response}')

        return {'content': content, 'steps': ['Rate Content']}
    
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, END, StateGraph
from src.backend.ai.summarize import Summarize
from src.backend.ai.llms import getAIModel
from typing import Literal

# -----------------------------------------------------------------------
def check_if_summary_needed(
        state: State,
    ) -> Literal['Summarize', 'Rate Content']:
        if len(state.get('content_pre').split()) > 500:
            return 'Summarize'
        return 'Rate Content'

# -----------------------------------------------------------------------
class Architecture:
     
     def __init__(self, model_name, temperature):
        llm = getAIModel(model_name=model_name, temperature=temperature)

        # Define a new graph
        workflow = StateGraph(state_schema=State)

        # Define the (single) node in the graph
        workflow.add_node("Summarize", Summarize(llm=llm))
        workflow.add_node("Rate Content", RateContent(llm=llm))
        #workflow.add_node("Add Citations", AddCitations(llm))

        workflow.add_conditional_edges(START, check_if_summary_needed)
        workflow.add_edge("Summarize", "Rate Content")

        # Add memory
        memory = MemorySaver()
        self.agent = workflow.compile(checkpointer=memory)


In [12]:
agent = Architecture(model_name='azure-gpt-4o', temperature=0).agent

In [13]:
import tqdm
import pandas as pd

# relevance, continuity, uniqueness, relevance of citation (hallucination), accuracy of the content+citation (hallucination), specificity of the content

from pathlib import Path

file_names = ['CRISPR-based editing for inherited blood disorders.md', 'Phthalates Toxicity.md', 'PFAS.md']

for file_name in file_names:
    print(f'Processing "{file_name}"...')
    section_pairs = getSectionPairsForComparison(file_name)
    content_pre_orig, content_pre_gen = '', ''
    rating_orig, rating_gen = [], []
    for section_header, [content_orig, content_gen] in tqdm.tqdm(section_pairs):
        if content_orig.strip() == '': continue
        rating_orig.append({'section_header': section_header} | agent.invoke({'current_section': section_header, 'content': content_orig, 'content_pre': content_pre_orig}, {"configurable": {"thread_id": "abc123"}})['content'])
        rating_gen.append({'section_header': section_header} | agent.invoke({'current_section': section_header, 'content': content_gen, 'content_pre': content_pre_gen}, {"configurable": {"thread_id": "abc123"}})['content'])

        content_pre_orig += f'{section_header}\n\n{content_orig}'
        content_pre_gen += f'{section_header}\n\n{content_gen}'

    rating_orig = pd.DataFrame(rating_orig)
    rating_gen = pd.DataFrame(rating_gen)

    rating = pd.merge(left=rating_orig, right=rating_gen, how='outer', on='section_header', suffixes=[' (Original)', ' (Generated)'])
    rating.to_csv(dir_evals / 'scores' / f'{Path(file_name).stem}.csv', index=None)

Processing "CRISPR-based editing for inherited blood disorders.md"...


100%|██████████| 19/19 [01:51<00:00,  5.89s/it]


Processing "Phthalates Toxicity.md"...


100%|██████████| 12/12 [00:48<00:00,  4.03s/it]


Processing "PFAS.md"...


100%|██████████| 30/30 [03:58<00:00,  7.96s/it]
